# Modulo 9 · Lezione 8 — Deployment Locale

**Progetto Finale: Sistema Intelligente di Predizione ed Esposizione di Modelli**

---

## Cosa vedremo oggi?

| # | Argomento |
|---|------|
| 1 | Produzione vs sviluppo |
| 2 | Servizio `systemd` su Linux |
| 3 | Docker: Dockerfile + immagine |
| 4 | Docker Compose — API + database |
| 5 | Checklist di produzione |

---
## 1. Produzione vs sviluppo

| Aspetto | Sviluppo | Produzione |
|---------|------------|------------|
| **Server** | `uvicorn --reload` | `gunicorn + uvicorn workers` |
| **Debug** | log dettagliati | solo errori |
| **Segreti** | `.env` locale | variabili d'ambiente del sistema |
| **Modelli** | file locale | volume Docker / S3 |
| **DB** | SQLite | PostgreSQL / cloud |
| **Restart** | manuale | automatico (systemd / Docker) |
| **HTTPS** | no | nginx + certificato |

In questa lezione affrontiamo il **deployment locale** (server Linux o macchina propria), che è il passo precedente al deployment nel cloud.

In [ ]:
from pathlib import Path
import subprocess, json

# Verificare che l'ambiente sia pronto
for p in ['api/main.py', 'models/rf_vinos_v1.joblib', 'requirements.txt']:
    existe = Path(p).exists()
    estado = '✓' if existe else '✗ MANCA'
    print(f"  {estado} {p}")

print("\nSe manca qualche file, esegui prima le lezioni precedenti.")

---
## 2. Servizio `systemd` su Linux

`systemd` è il sistema di avvio di Linux. Permette a un processo di partire automaticamente con il sistema, di riavviarsi se fallisce e di registrare i suoi log.

In [ ]:
# Generare il file di unit systemd
# NOTA: Per installarlo davvero servono permessi root su Linux
# Questo blocco genera il file affinché tu possa esaminarlo

import getpass, os

PROYECTO_DIR = Path.cwd().absolute()
VENV_DIR     = PROYECTO_DIR / '.venv'
USUARIO      = getpass.getuser()

systemd_unit = f"""[Unit]
Description=Wine Quality API
After=network.target

[Service]
Type=simple
User={USUARIO}
WorkingDirectory={PROYECTO_DIR}
Environment=PYTHONPATH={PROYECTO_DIR}
ExecStart={VENV_DIR}/bin/gunicorn api.main:app \\
    --workers 2 \\
    --worker-class uvicorn.workers.UvicornWorker \\
    --bind 0.0.0.0:8000 \\
    --log-level warning \\
    --access-logfile logs/access.log \\
    --error-logfile logs/error.log
Restart=always
RestartSec=5

[Install]
WantedBy=multi-user.target
"""

Path('docs/wine-api.service').write_text(systemd_unit)
print(systemd_unit)
print("\nSalvato in docs/wine-api.service")

In [ ]:
print("Comandi per installare il servizio (richiede root su Linux):")
print("""
# Copiare la unit nella directory di systemd
sudo cp docs/wine-api.service /etc/systemd/system/

# Ricaricare systemd affinché riconosca la nuova unit
sudo systemctl daemon-reload

# Abilitare il servizio (si avvia a ogni riavvio del sistema)
sudo systemctl enable wine-api

# Avviare ora
sudo systemctl start wine-api

# Verificare lo stato
sudo systemctl status wine-api

# Vedere i log in tempo reale
sudo journalctl -u wine-api -f

# Fermare / riavviare
sudo systemctl stop wine-api
sudo systemctl restart wine-api
"""
)

### Perché `gunicorn` invece del solo `uvicorn`?

- `uvicorn` è il server ASGI (asincrono). Eccellente, ma se il processo muore, l'API scompare.
- `gunicorn` è il **process manager**: gestisce più worker e li riavvia se falliscono.
- Combinazione standard: `gunicorn --worker-class uvicorn.workers.UvicornWorker`

In [ ]:
# Test di gunicorn in locale
import subprocess, time, requests

proc = subprocess.Popen(
    ['python', '-m', 'gunicorn', 'api.main:app',
     '--workers', '2',
     '--worker-class', 'uvicorn.workers.UvicornWorker',
     '--bind', '0.0.0.0:8002',
     '--log-level', 'warning'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)

if proc.poll() is None:
    r = requests.get('http://localhost:8002/health')
    print(f"✓ gunicorn avviato — health: {r.json()}")
    print(f"  2 worker attivi")
    proc.terminate(); proc.wait()
    print("✓ Server fermato")
else:
    stderr = proc.stderr.read().decode()
    print(f"Errore (gunicorn è installato? pip install gunicorn):\n{stderr[:300]}")

---
## 3. Docker: Dockerfile + immagine

In [ ]:
%%writefile Dockerfile
# Base: Python 3.11 slim (immagine piccola, senza GUI)
FROM python:3.11-slim

# Evitare interazione nelle installazioni
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

# Directory di lavoro dentro il container
WORKDIR /app

# Copiare solo requirements prima (sfrutta la cache dei layer Docker)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiare il codice sorgente
COPY api/ ./api/
COPY models/ ./models/

# Creare le directory dei dati a runtime
RUN mkdir -p data/raw data/processed data/db logs

# Esporre la porta
EXPOSE 8000

# Comando di avvio in produzione
CMD ["gunicorn", "api.main:app",
     "--workers", "2",
     "--worker-class", "uvicorn.workers.UvicornWorker",
     "--bind", "0.0.0.0:8000",
     "--log-level", "warning"]

In [ ]:
print("Comandi Docker di base:")
print("""
# Costruire l'immagine (etichettata come 'wine-api')
docker build -t wine-api .

# Eseguire il container
docker run -d \\
    --name wine-api-container \\
    -p 8000:8000 \\
    wine-api

# Vedere i log
docker logs wine-api-container -f

# Entrare nel container
docker exec -it wine-api-container bash

# Fermare ed eliminare
docker stop wine-api-container
docker rm wine-api-container

# Elencare le immagini
docker images
""")

In [ ]:
# Verificare se Docker è disponibile nel sistema
result = subprocess.run(['docker', '--version'], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✓ Docker disponibile: {result.stdout.strip()}")

    # Costruire l'immagine se Docker è installato
    print("\nCostruzione dell'immagine Docker... (può richiedere 2-5 minuti la prima volta)")
    build = subprocess.run(
        ['docker', 'build', '-t', 'wine-api:v1', '.'],
        capture_output=True, text=True
    )
    if build.returncode == 0:
        print("✓ Immagine costruita: wine-api:v1")
        # Dimensione dell'immagine
        size = subprocess.run(['docker', 'image', 'inspect', 'wine-api:v1',
                               '--format', '{{.Size}}'],
                              capture_output=True, text=True)
        mb = int(size.stdout.strip()) / 1024**2
        print(f"  Dimensione: {mb:.0f} MB")
    else:
        print(f"Errore nel build: {build.stderr[-500:]}")
else:
    print("Docker non è installato su questo sistema.")
    print("  Per installarlo: https://docs.docker.com/get-docker/")

---
## 4. Docker Compose — più servizi

In [ ]:
%%writefile docker-compose.yml
version: '3.9'

services:

  api:
    build: .
    image: wine-api:latest
    container_name: wine_api
    ports:
      - "8000:8000"
    volumes:
      # Rendere persistenti dati e modelli fuori dal container
      - ./data:/app/data
      - ./models:/app/models
      - ./logs:/app/logs
    environment:
      - PYTHONUNBUFFERED=1
      - LOG_LEVEL=warning
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 20s

  # Nginx come reverse proxy (gestisce HTTPS e bilanciamento)
  nginx:
    image: nginx:alpine
    container_name: wine_nginx
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/conf.d/default.conf:ro
    depends_on:
      api:
        condition: service_healthy
    restart: unless-stopped

volumes:
  data:
  models:

In [ ]:
%%writefile nginx.conf
upstream wine_api {
    server api:8000;
}

server {
    listen 80;
    server_name localhost;

    # Limitare la dimensione della richiesta
    client_max_body_size 1M;

    location / {
        proxy_pass         http://wine_api;
        proxy_set_header   Host $host;
        proxy_set_header   X-Real-IP $remote_addr;
        proxy_set_header   X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_read_timeout 60s;
    }

    # Endpoint di health check diretto (senza passare da FastAPI)
    location /nginx-health {
        return 200 'ok';
        add_header Content-Type text/plain;
    }
}

In [ ]:
print("Comandi Docker Compose:")
print("""
# Avviare tutti i servizi in background
docker compose up -d

# Vedere lo stato dei servizi
docker compose ps

# Vedere i log di tutti i servizi
docker compose logs -f

# Vedere i log solo dell'API
docker compose logs api -f

# Fermare senza eliminare i container
docker compose stop

# Fermare ed eliminare i container (i dati nei volumi persistono)
docker compose down

# Ricostruire l'immagine e riavviare
docker compose up -d --build

# Entrare nel container dell'API
docker compose exec api bash
""")

---
## 5. Checklist di produzione

In [ ]:
import requests, json, time
from pathlib import Path

def checklist_produccion(base_url: str = 'http://localhost:8000') -> dict:
    """Verifica i requisiti minimi di un deployment in produzione."""
    resultados = {}

    # 1. Health check
    try:
        r = requests.get(f'{base_url}/health', timeout=5)
        resultados['health_check'] = r.status_code == 200
    except Exception as e:
        resultados['health_check'] = False

    # 2. Documentazione accessibile
    try:
        r = requests.get(f'{base_url}/docs', timeout=5)
        resultados['docs_accesible'] = r.status_code == 200
    except Exception:
        resultados['docs_accesible'] = False

    # 3. La predizione funziona
    vino_test = {
        "tipo": "blanco", "fixed_acidity": 6.5, "volatile_acidity": 0.35,
        "citric_acid": 0.3, "residual_sugar": 7.0, "chlorides": 0.04,
        "free_sulfur_dioxide": 25.0, "total_sulfur_dioxide": 120.0,
        "density": 0.9942, "pH": 3.3, "sulphates": 0.55, "alcohol": 11.5
    }
    try:
        r = requests.post(f'{base_url}/predict', json=vino_test, timeout=10)
        res = r.json()
        resultados['prediccion_ok'] = (
            r.status_code == 200 and
            'prediccion' in res and
            'probabilidad' in res
        )
    except Exception:
        resultados['prediccion_ok'] = False

    # 4. La validazione degli errori funziona (422)
    try:
        r = requests.post(f'{base_url}/predict',
                          json={**vino_test, 'alcohol': 999}, timeout=5)
        resultados['validacion_errores'] = r.status_code == 422
    except Exception:
        resultados['validacion_errores'] = False

    # 5. File critici presenti
    archivos_criticos = [
        'api/main.py', 'models/rf_vinos_v1.joblib',
        'requirements.txt', 'Dockerfile'
    ]
    resultados['archivos_criticos'] = all(
        Path(a).exists() for a in archivos_criticos
    )

    # 6. Log configurati
    resultados['directorio_logs'] = Path('logs').is_dir()

    # 7. Prestazioni di base (<200ms per predizione)
    try:
        tiempos = []
        for _ in range(5):
            t0 = time.perf_counter()
            requests.post(f'{base_url}/predict', json=vino_test, timeout=10)
            tiempos.append((time.perf_counter() - t0) * 1000)
        p95 = sorted(tiempos)[4]
        resultados['rendimiento_ok'] = p95 < 200
        resultados['latencia_p95_ms'] = round(p95, 1)
    except Exception:
        resultados['rendimiento_ok'] = False

    return resultados

# Avviare il server ed eseguire la checklist
import subprocess, time
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'api.main:app', '--port', '8003', '--log-level', 'warning'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(2)

if proc.poll() is None:
    resultados = checklist_produccion('http://localhost:8003')

    print("CHECKLIST DI PRODUZIONE")
    print('=' * 40)
    aprobados = 0
    total = 0
    for check, valor in resultados.items():
        if check == 'latencia_p95_ms':
            print(f"  Latenza P95: {valor}ms")
            continue
        total += 1
        estado = '✓' if valor else '✗'
        if valor:
            aprobados += 1
        print(f"  {estado} {check.replace('_', ' ').capitalize()}")

    print(f"\nRisultato: {aprobados}/{total} controlli superati")
    if aprobados == total:
        print("✓ Sistema pronto per la produzione")
    else:
        print("✗ Rivedere i controlli falliti prima del deployment")

    proc.terminate(); proc.wait()
else:
    print("Errore nell'avvio del server")

---
## Pratica

### Esercizio 1 — Variabili d'ambiente

**Consegna:** Rifattorizza `api/main.py` per leggere la configurazione dalle variabili d'ambiente:

```python
import os
MODEL_PATH  = os.getenv('MODEL_PATH', 'models/rf_vinos_v1.joblib')
LOG_LEVEL   = os.getenv('LOG_LEVEL', 'warning')
PORT        = int(os.getenv('PORT', '8000'))
UMBRAL      = float(os.getenv('UMBRAL_CALIDAD', '0.5'))
```

Poi dimostra che puoi cambiare il modello senza toccare il codice:
```bash
MODEL_PATH=models/dl_clf_v1.keras uvicorn api.main:app
```

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

Aggiungi all'inizio di `api/main.py`:

```python
import os
from pydantic_settings import BaseSettings  # pip install pydantic-settings

class Settings(BaseSettings):
    model_path:       str   = 'models/rf_vinos_v1.joblib'
    log_level:        str   = 'warning'
    port:             int   = 8000
    umbral_calidad:   float = 0.5
    workers:          int   = 2

    class Config:
        env_file = '.env'  # carica da .env se esiste

settings = Settings()

# Usare settings nel codice:
# settings.model_path → percorso del modello
# settings.umbral_calidad → soglia di classificazione
```

Crea un `.env.example` (non il `.env` reale, che va in .gitignore):
```bash
# .env.example
MODEL_PATH=models/rf_vinos_v1.joblib
LOG_LEVEL=warning
PORT=8000
UMBRAL_CALIDAD=0.5
WORKERS=2
```

Testa senza modificare codice:
```bash
MODEL_PATH=models/dl_clf_v1.keras uvicorn api.main:app --port 8000
```
</details>

---
### Esercizio 2 — Immagine Docker ottimizzata

**Consegna:** Il Dockerfile che abbiamo generato include TensorFlow, che pesa ~500MB. Crea una versione ottimizzata per servire SOLO il modello Random Forest:

1. Crea `requirements_minimal.txt` senza tensorflow
2. Crea `Dockerfile.minimal` che usi questa immagine
3. Confronta la dimensione delle due immagini: `docker images`

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import subprocess
from pathlib import Path

# requirements_minimal.txt — senza tensorflow
reqs_minimal = """
fastapi>=0.110
uvicorn[standard]>=0.27
gunicorn>=21
pydantic>=2.5
scikit-learn>=1.4
numpy>=1.26
pandas>=2.2
joblib>=1.3
""".strip()
Path('requirements_minimal.txt').write_text(reqs_minimal)

# Dockerfile.minimal
dockerfile_min = '''FROM python:3.11-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
COPY requirements_minimal.txt .
RUN pip install --no-cache-dir -r requirements_minimal.txt
COPY api/ ./api/
COPY models/rf_vinos_v1.joblib ./models/
RUN mkdir -p data/db logs
EXPOSE 8000
CMD ["gunicorn", "api.main:app",
     "--workers", "2",
     "--worker-class", "uvicorn.workers.UvicornWorker",
     "--bind", "0.0.0.0:8000"]
'''
Path('Dockerfile.minimal').write_text(dockerfile_min)

# Costruire
result = subprocess.run(
    ['docker', 'build', '-f', 'Dockerfile.minimal', '-t', 'wine-api:minimal', '.'],
    capture_output=True, text=True
)
if result.returncode == 0:
    # Confrontare le dimensioni
    imgs = subprocess.run(['docker', 'images', 'wine-api'], capture_output=True, text=True)
    print(imgs.stdout)
else:
    print(result.stderr[-500:])
```
</details>

---
### Esercizio 3 — Script di deployment automatico

**Consegna:** Crea `scripts/deploy.sh` che automatizzi il deployment completo:

1. Verifica che Docker sia installato
2. Costruisce la nuova immagine
3. Ferma la versione precedente se in esecuzione
4. Avvia la nuova
5. Attende l'health check
6. Se l'health check fallisce, fa rollback all'immagine precedente

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```bash
#!/bin/bash
# scripts/deploy.sh — deployment con rollback automatico
set -e

IMAGE="wine-api"
CONTAINER="wine_api"
PORT="8000"
HEALTH_URL="http://localhost:${PORT}/health"

# 1. Verificare Docker
if ! command -v docker &> /dev/null; then
    echo "ERRORE: Docker non è installato" && exit 1
fi

# 2. Salvare la versione precedente
OLD_IMAGE=$(docker inspect ${CONTAINER} --format='{{.Config.Image}}' 2>/dev/null || echo "")
echo "Versione precedente: ${OLD_IMAGE:-'nessuna'}"

# 3. Costruire la nuova immagine
echo "Costruzione dell'immagine..."
docker build -t ${IMAGE}:new . || { echo "ERRORE nel build"; exit 1; }

# 4. Fermare ed eliminare il container precedente
docker stop ${CONTAINER} 2>/dev/null || true
docker rm   ${CONTAINER} 2>/dev/null || true

# 5. Avviare la nuova versione
echo "Avvio della nuova versione..."
docker run -d --name ${CONTAINER} -p ${PORT}:8000 \
    -v $(pwd)/data:/app/data \
    -v $(pwd)/models:/app/models \
    -v $(pwd)/logs:/app/logs \
    ${IMAGE}:new

# 6. Attendere l'health check
echo "In attesa dell'health check..."
for i in $(seq 1 12); do
    sleep 5
    if curl -sf ${HEALTH_URL} > /dev/null 2>&1; then
        echo "✓ Deployment riuscito su ${HEALTH_URL}"
        docker tag ${IMAGE}:new ${IMAGE}:latest
        exit 0
    fi
    echo "  Tentativo ${i}/12..."
done

# 7. Rollback
echo "ERRORE: Health check fallito. Rollback in corso..."
docker stop ${CONTAINER} && docker rm ${CONTAINER}
if [ -n "${OLD_IMAGE}" ]; then
    docker run -d --name ${CONTAINER} -p ${PORT}:8000 ${OLD_IMAGE}
    echo "Rollback completato a ${OLD_IMAGE}"
fi
exit 1
```
```bash
chmod +x scripts/deploy.sh && ./scripts/deploy.sh
```
</details>

---
## Chiusura del Modulo 9

Congratulazioni! Hai costruito un **Sistema Intelligente di Predizione ed Esposizione di Modelli** completo:

```
Lezione 1 — Dati:        Download Wine Quality UCI con integrità SHA256
Lezione 2 — Pulizia:     Outlier, encoding, feature engineering
Lezione 3 — SQL:         SQLite + SQLAlchemy + pandas ↔ SQL
Lezione 4 — ML:          Random Forest + GridSearchCV + joblib
Lezione 5 — DL:          TensorFlow/Keras clf + reg + callback
Lezione 6 — API:         FastAPI + Pydantic + uvicorn
Lezione 7 — Docs:        Docstring + README + OpenAPI + Git
Lezione 8 — Deployment:  systemd + Docker + Docker Compose
```

### Prossimi passi come AI Engineer

- **Cloud**: deployment su AWS/GCP/Azure (EC2, Cloud Run, App Service)
- **MLOps**: MLflow per gli esperimenti, DVC per il versionamento dei dati
- **Monitoraggio**: Prometheus + Grafana per le metriche in produzione
- **CI/CD**: GitHub Actions per test + deployment automatico
- **Scala**: Kubernetes per più repliche e alta disponibilità

---
## Riepilogo finale

```bash
# Sviluppo locale
uvicorn api.main:app --reload --port 8000

# Produzione con gunicorn
gunicorn api.main:app --workers 2 --worker-class uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000

# Docker
docker build -t wine-api .
docker run -d -p 8000:8000 wine-api

# Docker Compose
docker compose up -d
docker compose ps
docker compose logs -f
docker compose down

# systemd (Linux, richiede root)
sudo systemctl start wine-api
sudo systemctl enable wine-api
sudo journalctl -u wine-api -f
```